# Validation, Errors, and Tests
Validate one virtual trade and test the cash rule.

## 1. Start with a dictionary

In [ ]:
trade_data = {"symbol": "SPY", "amount": 1_000}
print(trade_data)

## 2. Define the request

In [ ]:
from pydantic import BaseModel, PositiveFloat

class TradeRequest(BaseModel):
    symbol: str
    amount: PositiveFloat

## 3. Validate a correct request

In [ ]:
trade = TradeRequest.model_validate(trade_data)
print(trade)

## 4. Deliberate failure: negative amount

In [ ]:
from pydantic import ValidationError

try:
    TradeRequest(symbol="SPY", amount=-10)
except ValidationError as error:
    print(error.errors()[0]["msg"])

## 5. Define a domain error

In [ ]:
class InsufficientCashError(Exception):
    pass

## 6. Check available cash

In [ ]:
def check_available_cash(trade: TradeRequest, cash: float) -> None:
    if trade.amount > cash:
        raise InsufficientCashError("Trade exceeds available cash")

check_available_cash(trade, 2_000)
print("Cash check passed")

## 7. Log a rejection

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

try:
    check_available_cash(trade, 500)
except InsufficientCashError as error:
    logging.warning("Rejected: %s", error)

## 8. Write a passing test

In [ ]:
def test_available_cash():
    request = TradeRequest(symbol="SPY", amount=100)
    check_available_cash(request, 500)


test_available_cash()
print("Passing test completed")

## 9. Test the exception

In [ ]:
def test_insufficient_cash():
    request = TradeRequest(symbol="SPY", amount=600)
    try:
        check_available_cash(request, 500)
        assert False, "Expected InsufficientCashError"
    except InsufficientCashError:
        pass

## 10. Run the tests directly

In [ ]:
test_available_cash()
test_insufficient_cash()
print("2 tests passed")

## 11. Final result

In [ ]:
validated = TradeRequest(symbol="GLD", amount=750)
check_available_cash(validated, 5_000)
print("Validated trade:", validated)

## Takeaways
- Pydantic checks data shape and constraints.
- Domain exceptions express business failures.
- Small assertions make behavior repeatable.